# Re-eval tier1 + tier4a at an alternate lambda (v3-r16)

Purpose: `src/pipeline.py`'s `sweep-gate` stage picks the LARGEST lambda that keeps
OOD CER within budget, not the cost/benefit elbow. On v3-r16's own sweep the elbow is
~0.5, not the 1.0 the pipeline picked (see `docs/finetune-results-report-v3.md` §4) --
matching a real risk found independently in the predecessor repo
(`docs/finetune-results-report-v2.md`): lambda=1.0 causes truncation/hallucination on
real audio that a bare CER number doesn't fully surface. `scripts/reeval_lambda.py`
re-evaluates tier1(test)+tier4a(real) at a chosen lambda without re-running the
already-known val/ood sweep -- this notebook runs it for **lambda=0.5** against the
existing v3-r16 checkpoint (no retraining, inference-only).

**Before running, attach as Kaggle Dataset inputs (Add Data) -- same two used for the
original training run:**
- `paid-dataset-v2`
- `real-meetings-bench`

**Also attach the v3-r16 run itself** (already on your local machine as
`Outputs/outputs_v3-r16.zip`, downloaded after the original training run) -- this
notebook does not retrain, it needs that run's `checkpoints/best/` adapter,
`config.json`, `validated_manifest.jsonl`, and `metrics/baseline.json` to re-evaluate
against. Either works, note which one so Cell 3 matches:
- **Kaggle Dataset**: upload the zip as-is -- Cell 3 will `unzip` it.
- **Kaggle Model** (e.g. under a "v3-r16" model, version 1): Kaggle extracts it for
  you at upload time -- it mounts as a plain folder under
  `/kaggle/input/models/<owner>/<model-slug>/<framework>/<variation>/<version>/...`,
  no zip file to unzip. Cell 3 has both paths; comment out the one you didn't use.

Run cells in order.

## 1. Clone / update the repo

In [ ]:
import os

# Force single-GPU -- see notebooks/run_pipeline.ipynb Cell 1 for why.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

REPO_URL = "https://github.com/egoist-minh/Reworkwhisper-finetune.git"
REPO_DIR = "/kaggle/working/Reworkwhisper-finetune"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt

## 2. Locate attached datasets

Confirm the exact mount paths before setting the paths below -- Kaggle slugs the
dataset name, so this can differ from what you expect.

In [ ]:
!ls -la /kaggle/input

## 3. Get the v3-r16 run onto disk and patch its config paths

The run's frozen `config.json` has the ORIGINAL training session's Kaggle mount
paths baked in (`/kaggle/input/datasets/winhkento/...`) -- a new session's mount
path can differ, so patch `data.dataset_path` / `data.real_bench_path` to match
what Cell 2 printed before running the script. Also note: the run's internal
folder is `v0-r16` (the `run_id` the original notebook run used before its output
folder was renamed locally to `v3-r16` for clarity -- cosmetic only, see
`docs/finetune-results-report-v3.md`) -- renamed to `v3-r16` here to match.

In [ ]:
import json
from pathlib import Path

DATASET_PATH = "/kaggle/input/paid-dataset-v2"         # edit to match Cell 2's listing
REAL_BENCH_PATH = "/kaggle/input/real-meetings-bench"  # edit to match Cell 2's listing

!mkdir -p outputs
!rm -rf outputs/v3-r16

# --- option A: uploaded as a Kaggle Dataset (a .zip to unzip) ---
# RUN_ARCHIVE = "/kaggle/input/v3-r16-checkpoint/outputs_v3-r16.zip"  # edit to match Cell 2's listing
# !unzip -q -o {RUN_ARCHIVE} -d .
# !mv outputs/v0-r16 outputs/v3-r16

# --- option B: uploaded as a Kaggle Model (already extracted, no zip) ---
RUN_SRC = "/kaggle/input/models/winhkento/v3-r16/other/default/1/outputs/v0-r16"  # edit to match the MODELS tree in the right-hand panel
!cp -r {RUN_SRC} outputs/v3-r16

RUN_DIR = Path("outputs/v3-r16")
cfg_path = RUN_DIR / "config.json"
cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
cfg["data"]["dataset_path"] = DATASET_PATH
cfg["data"]["real_bench_path"] = REAL_BENCH_PATH
cfg_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding="utf-8")
print("patched config.json:", cfg["data"]["dataset_path"], "|", cfg["data"]["real_bench_path"])


## 4. Run the re-eval at lambda=0.5

Inference-only -- no training, no HF push. Writes
`outputs/v3-r16/metrics/gate_results_lambda0.5.json` and
`outputs/v3-r16/audit/predictions_{tier1_in_domain,tier4a_real}_lambda0.5.csv`.
Does not touch the run's official `metrics/gate_results.json` (the lambda=1.0
record already gated).

In [ ]:
!python -m scripts.reeval_lambda outputs/v3-r16 0.5

## 5. Read the result

In [ ]:
import json
print(json.dumps(json.load(open("outputs/v3-r16/metrics/gate_results_lambda0.5.json")), indent=2, ensure_ascii=False))


## 6. Download the new evidence

Only the lambda=0.5 artifacts are new -- zip just those plus the result json,
not the whole run folder again.

In [ ]:
!zip -q v3-r16_lambda0.5.zip     outputs/v3-r16/metrics/gate_results_lambda0.5.json     outputs/v3-r16/audit/predictions_tier1_in_domain_lambda0.5.csv     outputs/v3-r16/audit/predictions_tier4a_real_lambda0.5.csv

from IPython.display import FileLink
FileLink("v3-r16_lambda0.5.zip")


## 7. HF token (only needed if you intend to push in the next section)

Add `HF_TOKEN` under this notebook's Add-ons -> Secrets first. Never hardcode the
token here.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print(f"No HF_TOKEN secret configured ({e}) -- required for the push cell below")


## 8. Push the lambda=0.5 adapter -- manual override, read before running

**This does not check `overall_pass`.** `src/pipeline.py`'s automatic push only
fires when the sweep-gate's own `overall_pass` is true; this cell calls
`src.hub.push_adapter` directly, the same "manual override, gate stays honest"
path already used for a prior run (see `docs/finetune-results-report-v3.md` --
push is deadline-driven, not gate-pass-driven). The model card written to the
pushed repo's `README.md` will show the REAL tier1/tier2/tier4a numbers from
this lambda, including a FAIL/INCONCLUSIVE tier4a if that's what Cell 6 showed --
this cell does not hide or reinterpret that.

Only run this after reading Cell 6's output yourself. Set `REPO_ID` and flip
`CONFIRM_OVERRIDE = True` -- both default to blocking the push so a blind
"run all" cannot publish anything.

In [ ]:
import csv
import json
from pathlib import Path

REPO_ID = None             # e.g. "your-username/phowhisper-lora-v3-r16-lambda0.5"
PRIVATE = True
CONFIRM_OVERRIDE = False   # you must flip this to True after reading Cell 6's output yourself

RUN_DIR = Path("outputs/v3-r16")
LAM = 0.5

if not CONFIRM_OVERRIDE or not REPO_ID:
    raise RuntimeError("Set REPO_ID and CONFIRM_OVERRIDE=True (after reading Cell 6's "
                        "output) before running this cell -- both default to blocking the push.")

# Assemble the real gate_results for this lambda: tier1/tier4a from Cell 4's re-eval,
# tier2 from lambda_sweep.csv (already known for every lambda in the grid -- no rerun
# needed, see docs/finetune-results-report-v3.md SS4).
reeval = json.loads((RUN_DIR / f"metrics/gate_results_lambda{LAM}.json").read_text(encoding="utf-8"))
baseline = json.loads((RUN_DIR / "metrics/baseline.json").read_text(encoding="utf-8"))
cfg = json.loads((RUN_DIR / "config.json").read_text(encoding="utf-8"))

with open(RUN_DIR / "metrics/lambda_sweep.csv", encoding="utf-8") as f:
    sweep_rows = {float(r["lambda"]): r for r in csv.DictReader(f)}
ood_cer = float(sweep_rows[LAM]["ood_cer"])
tier2_bound = baseline["cer_ood"] + cfg["sweep"]["ood_cer_budget"]

gate_results = {
    "tier1_in_domain": reeval["tier1_in_domain"],
    "tier2_ood": {"cer": ood_cer, "bound": tier2_bound, "pass": ood_cer <= tier2_bound},
    "tier4a_real": reeval["tier4a_real"],
}
gate_results["overall_pass"] = all(t.get("pass") is not False for t in gate_results.values())
print(json.dumps(gate_results, indent=2, ensure_ascii=False))

from src import compat
compat.apply()
from src.hub import push_adapter

adapter_dir = RUN_DIR / f"adapter_lambda{LAM}"
url = push_adapter(adapter_dir, REPO_ID, private=PRIVATE, gate_results=gate_results)
print(f"pushed to {url}")
